<b>Run Alignment</b>

In [2]:
%%bash

SCRIPT_DIR="/Users/wuchh/Local/repositories/tree-alignment/O-MLTED"
TREE_ALIGNMENT_DIR="/Users/wuchh/Downloads/126_tx_dataset-analysis-22may2026/vm/tree-alignment"
ALIGNMENT_SCRIPT_DIR="/Users/wuchh/Local/repositories/tree-alignment/workspace/biowulf/python/"

# Colors
readonly GREEN='\033[0;32m'
readonly RED='\033[0;31m'
# readonly YELLOW='\033[1;33m'
readonly YELLOW_BG='\033[43;30;1m'
readonly NC='\033[0m'

success_count=0

for dirs in "$TREE_ALIGNMENT_DIR"/*; do
    [[ -d "$dirs" ]] || continue
    treeset="${dirs##*/}"

    echo "Processing treeset: $treeset"
    
    if [[ ! -d "${dirs}/conipher" || ! -d "${dirs}/pairtree" ]]; then
        echo -e "   ${RED}✗ ${treeset}: missing conipher or pairtree folder${NC}"
        continue
    fi
    
    conipher_tree="${dirs}/conipher/${treeset}.conipher.txt"
    [[ -f "$conipher_tree" ]] || { 
        echo -e "   ${RED}✗ ${treeset}: conipher file not found${NC}"
        continue
    }

    best_value=""
    best_file=""
    
    for pairtree_file in "${dirs}/pairtree/"*-soln_0.txt; do
        [[ -f "$pairtree_file" ]] || continue
        
        output=$(python "$SCRIPT_DIR/OMLTED.py" "$conipher_tree" "$pairtree_file" 2>&1)
        
        if [[ $? -eq 0 ]]; then
            omlted_line=$(echo "$output" | grep "Normalized OMLTED:")
            if [[ -n "$omlted_line" ]]; then
                value=$(echo "$omlted_line" | grep -o '[0-9]*\.[0-9]*')
                echo -e "  ${GREEN}✓ Success: $(basename "$pairtree_file") ~ $omlted_line${NC}"
                
                # Track best (lowest) value
                if [[ -z "$best_value" ]] || (( $(echo "$value <= $best_value" | bc -l) )); then
                    best_value="$value"
                    best_file="$(basename "$pairtree_file")"
                fi
            else
                echo -e "  ${GREEN}✓ Success: $(basename "$pairtree_file") ~ No OMLTED found${NC}"
            fi
        else
            echo -e "  ${RED}✗ Failed: $(basename "$pairtree_file")${NC}"
        fi
    done
    
    if [[ -n "$best_file" ]]; then
        echo -e "  ${YELLOW_BG} ! Best: $best_file ~ $best_value ${NC}"
        file_alignment_name=$(basename "${best_file%.*}")
        #echo "   → RUNNING: "
        python $SCRIPT_DIR/OMLTED.py $conipher_tree ${dirs}/pairtree/${best_file} -o ${TREE_ALIGNMENT_DIR}/${treeset}/edit_sequence_${treeset}.txt > /dev/null 2>&1
        echo "   Applying edits to align input trees ... "
        python $ALIGNMENT_SCRIPT_DIR/applyedits.py -o ${TREE_ALIGNMENT_DIR}/${treeset}/${file_alignment_name}.aligned.tsv $conipher_tree ${dirs}/pairtree/${best_file} ${TREE_ALIGNMENT_DIR}/${treeset}/edit_sequence_${treeset}.txt
        ((success_count++))
    fi
done
echo "${success_count}/126"

Processing treeset: CRUK0003
  ✓ Success: CRUK0003.pairtree-soln_0.txt ~ Normalized OMLTED: 0.402
   ! Best: CRUK0003.pairtree-soln_0.txt ~ 0.402 
   Applying edits to align input trees ... 
   ✓ Trees are isomorphic: True
Processing treeset: CRUK0004
  ✓ Success: CRUK0004.pairtree-soln_0.txt ~ Normalized OMLTED: 0.275
   ! Best: CRUK0004.pairtree-soln_0.txt ~ 0.275 
   Applying edits to align input trees ... 
   ✓ Trees are isomorphic: True
Processing treeset: CRUK0009
  ✓ Success: CRUK0009.pairtree-soln_0.txt ~ Normalized OMLTED: 0.472
   ! Best: CRUK0009.pairtree-soln_0.txt ~ 0.472 
   Applying edits to align input trees ... 
   ✓ Trees are isomorphic: False
Processing treeset: CRUK0010
  ✓ Success: CRUK0010.pairtree-soln_0.txt ~ Normalized OMLTED: 0.385
   ! Best: CRUK0010.pairtree-soln_0.txt ~ 0.385 
   Applying edits to align input trees ... 
   ✓ Trees are isomorphic: True
Processing treeset: CRUK0013
  ✓ Success: CRUK0013.pairtree-soln_0.txt ~ Normalized OMLTED: 0.198
   ! Best